In [ ]:
library(rtracklayer)
library(ggplot2)
library(Seurat)

getwd()
dir.create("figures_10xPBMC")
dir.create("data")


colorTools = c("SoloTE"="#A4DD9B", #6fc69d",
               "Stellarscope"="#4f5d93",
               "STARsolo"="#f0df93")

dataset_id <- "10xPBMC"

In [ ]:
# theme_paper <- function(base_size = 17, base_family = "") {
#   theme_minimal(base_size = base_size, base_family = base_family) +
#     theme(
#       plot.title = element_text(face = "bold", size = base_size + 2, hjust = 0.5),
#       axis.title = element_text(size = base_size),
#       axis.text  = element_text(size = base_size * 0.9),
#       legend.title = element_text(size = base_size),
#       legend.text  = element_text(size = base_size * 0.9),
#       panel.grid.major = element_line(linewidth = 0.4),
#       panel.grid.minor = element_blank(),
#       plot.margin = margin(10, 10, 10, 10)
#     )
# }

# theme_set(theme_paper())

In [ ]:
#save.image("workspaces/afterCorrelations_wSTAR.RData")
load("workspaces/afterCorrelations_10xPBMC_wSTAR_noPlatelet.RData")
gc()


# Gene Intersections

In [ ]:
library(rtracklayer)
# import the annotation used for snakemake
gene_annotation <- rtracklayer::import("/mnt/TEdataStorage_2T/snakemake_data/references/hg38/gencode.v30.annotation.gtf.gz")
names(mcols(gene_annotation))

In [ ]:
gene_annotation_df <- as.data.frame(gene_annotation)

# Extract only exon entries
exons <- gene_annotation[gene_annotation$type == "exon"]
names(exons) <- exons$gene_name

In [ ]:
# keep only TEs that are in both SoloTE and Stellarscope
annotation_common <- na.omit(conversionTable)

cat("Classes not in SoloTE annotation: ", 
    paste(setdiff(unique(conversionTable$class), unique(annotation_common$class)), collapse = ", "), 
    "\n")

In [ ]:
objTElist <- list()

objTElist[["SoloTE"]] <- objTE_soloTE
objTElist[["Stellarscope"]] <- objTE_stellarscope
objTElist[["STARsolo"]] <- objTE_STARsolo

objTElist

In [ ]:
# Compute overlaps to exons and save lists of loci
featuresList <- list()

bedList <- list()
grList <- list()
exonOverlapList <- list()
toolOverlappingGrList <- list()
toolOverlappingList <- list()

for(tool in names(objTElist)){
    if(tool == "SoloTE"){
        bedList[[tool]] <- conversionTable[conversionTable$soloteID %in% Features(objTElist[[tool]]), 
                            c("chr","start","end","strand","stellarscopeID")]
        print(nrow(bedList[[tool]]))
    }else{
        bedList[[tool]] <- conversionTable[conversionTable$stellarscopeID %in% Features(objTElist[[tool]]), 
                            c("chr","start","end","strand","stellarscopeID")]
        print(nrow(bedList[[tool]]))
    }

    bedList[[tool]] <- bedList[[tool]][!duplicated(bedList[[tool]]$stellarscopeID),]
    bedList[[tool]] <- bedList[[tool]][!is.na(bedList[[tool]]$stellarscopeID),]
    featuresList[[tool]] <- bedList[[tool]]$stellarscopeID
    print(nrow(bedList[[tool]]))
    rownames(bedList[[tool]]) <- bedList[[tool]]$stellarscopeID
    # turn into GR object
    grList[[tool]] <- makeGRangesFromDataFrame(bedList[[tool]])
    exonOverlapList[[tool]] <- findOverlaps(grList[[tool]], exons, minoverlap = 10)

    toolOverlappingGrList[[tool]] <- grList[[tool]][unique( exonOverlapList[[tool]]@from)] 
    toolOverlappingList[[tool]] <- rownames(as.data.frame(toolOverlappingGrList[[tool]]))
    #print(as.data.frame(toolOverlappingList[[tool]]))
}


In [ ]:
colorTools

In [ ]:
library(UpSetR)
options(repr.plot.width=9, repr.plot.height=6)

ph <- UpSetR::upset(fromList(toolOverlappingList), 
    sets=names(toolOverlappingList), sets.bar.color=colorTools,
    keep.order = TRUE,
    text.scale = c(2, 2, 2, 1.75, 2.75, 2.5), 
    order.by=c("freq"),
    point.size=4, mb.ratio = c(0.63, 0.37),
    set_size.show=F, set_size.scale_max= max(lengths(toolOverlappingList))+800, 
    sets.x.label="N. loci intersecting \n gene exons",
    mainbar.y.label="Intersection size")

pdf(paste0("figures_", dataset_id, "/NlociIntersectingGeneExons_upset.pdf"), width=9, height=6)
ph
dev.off()
ph

In [ ]:
df <- NULL

for(tool in names(objTElist)){
    
    loci <- bedList[[tool]]$stellarscopeID
    lociOverlappingGenes <- toolOverlappingList[[tool]]

    df <- rbind(df, c(tool, length(loci), length(lociOverlappingGenes)))

}

df <-  as.data.frame(df)
df

In [ ]:
library(dplyr)
library(purrr)
library(tidyr)

make_membership_table <- function(all_lists, gene_lists=NULL) {
  
  # All loci that appear in at least one tool
  all_loci <- unique(unlist(all_lists))
  
  # Presence/absence table (automatic)
  df <- tibble(locus = all_loci)
  
  for (nm in names(all_lists)) {
    df[[nm]] <- df$locus %in% all_lists[[nm]]
  }
  
  # Optional: add gene-overlap column
  if (!is.null(gene_lists)) {
    gene_loci <- unique(unlist(gene_lists))
    df$gene_overlap <- df$locus %in% gene_loci
  }
  
  df
}

df <- make_membership_table(featuresList, toolOverlappingList)
head(df)


In [ ]:
df <- df %>%
  mutate(
    group = pmap_chr(select(., SoloTE:STARsolo), ~ {
      here <- c(...)
      tools <- names(here)[here]
      if (length(tools) == 0) return(NA_character_)
      paste(tools, collapse = "&")
    })
  )


In [ ]:
summary_df <- df %>%
  group_by(group) %>%
  summarise(
    N = n(),
    gene_N = sum(gene_overlap),
    percent_gene = round(100 * gene_N / N, 1),
    .groups = "drop"
  ) %>%
  arrange(desc(N))

In [ ]:
lengths(toolOverlappingList)
summary_df

In [ ]:
mix_colors <- function(cols) {
  cols_rgb <- t(col2rgb(cols))
  avg_rgb <- colMeans(cols_rgb)
  rgb(avg_rgb[1], avg_rgb[2], avg_rgb[3], maxColorValue = 255)
}

library(purrr)

# 1-way (original tool colors)
color_1way <- colorTools

# 2-way mixes
color_2way <- list(
  "SoloTE&Stellarscope" = mix_colors(colorTools[c("SoloTE","Stellarscope")]),
  "SoloTE&STARsolo"     = mix_colors(colorTools[c("SoloTE","STARsolo")]),
  "Stellarscope&STARsolo" = mix_colors(colorTools[c("Stellarscope","STARsolo")])
)

# 3-way mix
color_3way <- c(
  "SoloTE&Stellarscope&STARsolo" =
    mix_colors(colorTools[c("SoloTE","Stellarscope","STARsolo")])
)

# Combine everything
intersection_colors <- c(color_1way, color_2way, color_3way)
intersection_colors


In [ ]:
summary_df

In [ ]:
options(repr.plot.width=9, repr.plot.height=4)
library(ggpubr)

df <- summary_df
# Step 1: compute size
df$size <- sapply(df$group, function(x) length(strsplit(x, "&")[[1]]))

# Step 2: get **unique groups** and sort by size
unique_groups <- unique(df$group)
group_levels <- unique_groups[order(sapply(unique_groups, function(x) length(strsplit(x, "&")[[1]])))]

# descending within each size
df <- df %>%
  arrange(-size, percent_gene)  # or arrange(size, percent) for ascending

df$group <- factor(df$group, levels = df$group)  # preserve new order


ggplot(df , aes(y = group, x = percent_gene, fill = group)) +
  geom_col(alpha=1) +
  scale_fill_manual(values = colorTools, na.value = "grey76") +
  xlab("% of TEs overlapping exons") +
  ylab("TEs detected by:") +
  xlim(0,60) +
  theme_pubclean() +
  theme(text=element_text(size=22), 
        axis.text=element_text(size=20), 
        legend.position = "none"
        ) 
ggsave(paste0("figures_", dataset_id, "/percTEsOverlappingExons.pdf"), width=8, height=6)


In [ ]:
options(repr.plot.width=9, repr.plot.height=4)

df <- summary_df
# Step 1: compute size
df$size <- sapply(df$group, function(x) length(strsplit(x, "&")[[1]]))

# Step 2: get **unique groups** and sort by size
unique_groups <- unique(df$group)
group_levels <- unique_groups[order(sapply(unique_groups, function(x) length(strsplit(x, "&")[[1]])))]

# descending within each size
df <- df %>%
  arrange(-size, percent_gene)  # or arrange(size, percent) for ascending

df$group <- factor(df$group, levels = df$group)  # preserve new order


ggplot(df , aes(y = group, x = percent_gene, fill = group)) +
  geom_col() +
  scale_fill_manual(values = colorTools, na.value = "grey76") +
  xlab("% of TEs overlapping exons") +
  ylab("TEs detected by:") +
  xlim(0,100) +
  theme_pubclean() +
  theme(text=element_text(size=22), 
        axis.text=element_text(size=20), 
        legend.position = "none"
        ) 
ggsave(paste0("figures_", dataset_id, "/percTEsOverlappingExons_max100.pdf"), width=8, height=4)


In [ ]:

# descending within each size
df <- df %>%
  arrange(-size, gene_N)  # or arrange(size, percent) for ascending

df$group <- factor(df$group, levels = df$group)  # preserve new order
ggplot(df , aes(y = group, x = gene_N, fill = group)) +
  geom_col() +
  scale_fill_manual(values = colorTools, na.value = "grey76") +
  xlab("N. TEs overlapping exons") +
  ylab("TEs detected by:") +
  theme_pubclean() +
  theme(text=element_text(size=20), 
        axis.text=element_text(size=20), 
        legend.position = "none",
        plot.margin = margin(t = 10, r = 30, b = 10, l = 10)  # top, right, bottom, left
        ) 
ggsave(paste0("figures_", dataset_id, "/nTEsOverlappingExons.pdf"), width=8, height=4)
